<a href="https://colab.research.google.com/github/GFDRR/urban_validation/blob/main/notebooks/99_check_data_inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reference dataset inventory and validation

This notebook inventories a Google Drive folder containing reference building-outline datasets, matches those folders to dataset codes listed in a Google Sheet, and writes summary tables that help verify which datasets are present and usable.

## What this notebook does

The notebook performs five main steps:

1. **Connect to Google Drive and Google Sheets**
   - Mounts your Google Drive in Colab
   - Authenticates using your current Colab Google account
   - Reads metadata from a target Drive folder and a worksheet in a Google Sheet

2. **Inventory the Drive folder**
   - Recursively scans the folder tree under a given Drive folder ID
   - Records folders, files, hierarchy level, file sizes, and derived dataset folder names

3. **Read the reference dataset table**
   - Loads the worksheet as a pandas DataFrame
   - Resolves the configured dataset-code and quality columns, allowing for minor differences in capitalization, spaces, hyphens, and underscores

4. **Match dataset codes to dataset folders**
   - Compares dataset codes from the sheet to top-level folder names in Drive
   - Uses normalized matching rules rather than exact string equality only
   - Attempts to tolerate punctuation differences, wrapper words, and some version-like suffixes
   - Selects the best-scoring folder match for each row, while also flagging ambiguous cases

5. **Write output tables**
   - A dataset-level summary
   - A file-level manifest
   - Validation outputs for unmatched datasets, duplicate dataset codes, and ambiguous duplicate matches
   - An optional dataset-level AOI/reference GeoJSON listing

## Matching logic

Dataset matching is heuristic, not exact. Folder names and dataset codes are normalized before comparison. The matching logic currently tries to handle:

- case differences
- spaces, underscores, and hyphens
- punctuation differences
- some wrapper prefixes/suffixes such as `dataset` or `data`
- some trailing version-like suffixes such as `v2`, `rev1`, or date-like variants

If multiple candidate folders match the same dataset code, the notebook keeps the highest-scoring match in the main summary and records the other candidates in the duplicate-matches output.

## Main assumptions

This notebook assumes that:

- each dataset is represented by a **top-level folder** directly under the configured Drive folder
- the Google Sheet contains one row per dataset reference entry
- the configured dataset-code column contains the identifiers intended to match Drive folder names
- the configured quality column contains values that can be interpreted as yes/no quality flags
- AOI and reference files follow naming conventions such as:
  - `*_aoi.geojson`
  - `*_ref.geojson`
  - `*_hotosm.geojson`

## Limitations

This notebook has several practical limitations:

- matching is rule-based and may still fail for unusual folder naming conventions
- ambiguous matches are flagged, but may still require manual review
- duplicate dataset codes in the sheet are reported, but not automatically resolved
- the notebook assumes that top-level folder names are the correct matching unit
- very large Drive inventories may take time to scan
- file naming conventions for AOI/reference outputs are assumed rather than inferred semantically

## Suggested future improvements

Possible improvements include:

- adding fuzzy string matching with review thresholds
- adding support for shared drives and larger inventories with stronger retry logic
- making wrapper/version token handling configurable
- logging match decisions more explicitly for reproducibility
- validating expected output files per dataset
- turning the notebook into a small reusable Python package plus a thin Colab wrapper

## Outputs

The notebook writes the following CSV files to the configured output directory:

- dataset summary
- file manifest
- unmatched datasets
- duplicate dataset codes
- duplicate matches
- optional AOI/reference GeoJSON listing

All outputs are written to a user-configurable folder in mounted Google Drive.


In [1]:
%pip -q install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas

In [2]:
# Mount Google Drive
from google.colab import drive  # type: ignore
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Authenticate with your current Colab Google account (no service-account JSON needed)
from google.colab import auth  # type: ignore
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

def build_drive_service():
    creds, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/drive.readonly"]
    )
    return build("drive", "v3", credentials=creds, cache_discovery=False)

def build_sheets_service():
    creds, _ = google.auth.default(
        scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )
    return build("sheets", "v4", credentials=creds, cache_discovery=False)

drive_service = build_drive_service()
sheets_service = build_sheets_service()
print("Authenticated successfully.")


Authenticated successfully.


In [4]:
# --- Configuration: edit only this cell for most use cases ---
from pathlib import Path

CONFIG = {
    # Required inputs
    "drive_folder_id": "13_18VeKUoaXijfa-G47glSNtrEn9-Rnu",
    "sheet_id": "1gJfCSGxcYb5z7JTkqeagWDlV6x-BtA9kZUdzp9HDZ9g",
    "worksheet": "Reference Dataset overview",

    # Worksheet columns
    "dataset_code_col": "Dataset Code",
    "quality_col": "Suitable (yes/N)",

    # Processing options
    "high_quality_only": True,

    # Output location in mounted Google Drive
    "output_drive_dir": "/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory",

    # Output filenames
    "summary_filename": "reference_download_status.csv",
    "files_filename": "reference_download_file_manifest.csv",
    "unmatched_filename": "reference_unmatched_datasets.csv",
    "duplicate_matches_filename": "reference_duplicate_matches.csv",
    "duplicate_codes_filename": "reference_duplicate_dataset_codes.csv",
    "geojson_listing_filename": "dataset_aoi_reference_list.csv",
}

def validate_config(config: dict) -> None:
    required_keys = [
        "drive_folder_id",
        "sheet_id",
        "worksheet",
        "dataset_code_col",
        "quality_col",
        "output_drive_dir",
    ]
    missing = [k for k in required_keys if k not in config or str(config[k]).strip() == ""]
    if missing:
        raise ValueError(f"Missing required CONFIG values: {missing}")

    if not str(config["output_drive_dir"]).startswith("/content/drive/"):
        raise ValueError(
            "CONFIG['output_drive_dir'] must point to mounted Google Drive, "
            "for example '/content/drive/MyDrive/...'"
        )

validate_config(CONFIG)

OUTPUT_DIR = Path(CONFIG["output_drive_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_OUT = OUTPUT_DIR / CONFIG["summary_filename"]
FILES_OUT = OUTPUT_DIR / CONFIG["files_filename"]
UNMATCHED_OUT = OUTPUT_DIR / CONFIG["unmatched_filename"]
DUPLICATE_MATCHES_OUT = OUTPUT_DIR / CONFIG["duplicate_matches_filename"]
DUPLICATE_CODES_OUT = OUTPUT_DIR / CONFIG["duplicate_codes_filename"]
GEOJSON_LIST_OUT = OUTPUT_DIR / CONFIG["geojson_listing_filename"]

print("Configuration loaded:")
print(f"  Drive folder ID     : {CONFIG['drive_folder_id']}")
print(f"  Sheet ID            : {CONFIG['sheet_id']}")
print(f"  Worksheet           : {CONFIG['worksheet']}")
print(f"  Dataset code column : {CONFIG['dataset_code_col']}")
print(f"  Quality flag column : {CONFIG['quality_col']}")
print(f"  High-quality only   : {CONFIG['high_quality_only']}")
print(f"  Output directory    : {OUTPUT_DIR}")


Configuration loaded:
  Drive folder ID     : 13_18VeKUoaXijfa-G47glSNtrEn9-Rnu
  Sheet ID            : 1gJfCSGxcYb5z7JTkqeagWDlV6x-BtA9kZUdzp9HDZ9g
  Worksheet           : Reference Dataset overview
  Dataset code column : Dataset Code
  Quality flag column : Suitable (yes/N)
  High-quality only   : True
  Output directory    : /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory


In [5]:
# Imports + helpers
from __future__ import annotations

from collections import deque
from typing import Iterable
import re
import unicodedata

import pandas as pd
from IPython.display import display

FOLDER_MIME = "application/vnd.google-apps.folder"

WRAPPER_PREFIXES = {
    "dataset", "datasets", "data", "ds", "folder", "files", "download", "downloads",
    "input", "inputs", "source", "sources"
}
WRAPPER_SUFFIXES = {
    "dataset", "datasets", "data", "folder", "folders", "file", "files", "archive",
    "backup", "copy", "copies", "raw", "processed", "proc", "final", "latest",
    "updated", "update", "export", "exports", "deliverable", "delivery"
}
VERSION_TOKEN_RE = re.compile(
    r"^(?:v(?:er(?:sion)?)?|rev|r)\d+[a-z]*$|^\d{8}$|^\d{6}$|^\d{4}$"
)

def ascii_lower(value) -> str:
    value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return value.lower().strip()

def tokenize(value) -> list[str]:
    return re.findall(r"[a-z0-9]+", ascii_lower(value))

def normalize_join(tokens: Iterable[str]) -> str:
    return "".join(tok for tok in tokens if tok)

def is_version_token(token: str) -> bool:
    return bool(VERSION_TOKEN_RE.match(token)) or token in {"final", "latest", "new"}

def strip_outer_tokens(tokens: list[str]) -> list[str]:
    tokens = list(tokens)
    while tokens and tokens[0] in WRAPPER_PREFIXES:
        tokens = tokens[1:]
    while tokens and (tokens[-1] in WRAPPER_SUFFIXES or is_version_token(tokens[-1])):
        tokens = tokens[:-1]
    return tokens

def token_subsequence(needle: list[str], haystack: list[str]) -> bool:
    if not needle or len(needle) > len(haystack):
        return False
    n = len(needle)
    for i in range(len(haystack) - n + 1):
        if haystack[i:i+n] == needle:
            return True
    return False

def analyze_name(value: str) -> dict:
    tokens = tokenize(value)
    core_tokens = strip_outer_tokens(tokens)
    return {
        "raw": value,
        "tokens": tokens,
        "core_tokens": core_tokens,
        "norm": normalize_join(tokens),
        "core_norm": normalize_join(core_tokens) or normalize_join(tokens),
    }

def parse_quality_flag(value) -> bool:
    if pd.isna(value):
        return False
    value = ascii_lower(value)
    yes_values = {"yes", "y", "true", "1", "high", "suitable"}
    no_values = {"no", "n", "false", "0", "low", "unsuitable"}
    if value in yes_values:
        return True
    if value in no_values:
        return False
    if "yes" in value or "suitable" in value or "high" in value:
        return True
    if value == "n" or "no" in value or "unsuitable" in value:
        return False
    return False

def score_folder_match(dataset_code: str, folder_name: str) -> tuple[int, str] | tuple[None, None]:
    code = analyze_name(dataset_code)
    folder = analyze_name(folder_name)

    if not code["norm"] or not folder["norm"]:
        return None, None

    if code["norm"] == folder["norm"]:
        return 100, "exact_normalized"

    if code["core_norm"] == folder["core_norm"]:
        return 96, "exact_core"

    if code["core_tokens"] and token_subsequence(code["core_tokens"], folder["tokens"]):
        return 90, "core_token_subsequence"

    if code["tokens"] and token_subsequence(code["tokens"], folder["tokens"]):
        return 88, "token_subsequence"

    if len(code["core_norm"]) >= 5 and code["core_norm"] in folder["core_norm"]:
        return 84, "core_substring"

    if len(code["norm"]) >= 6 and code["norm"] in folder["norm"]:
        return 82, "normalized_substring"

    if len(folder["core_norm"]) >= 5 and folder["core_norm"] in code["core_norm"]:
        return 78, "reverse_core_substring"

    return None, None

def _norm_colname(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\s_\-]+", "", s)
    return s

def resolve_column_name(df: pd.DataFrame, requested: str) -> str:
    if requested in df.columns:
        return requested

    requested_norm = _norm_colname(requested)
    candidates = [c for c in df.columns if _norm_colname(c) == requested_norm]

    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise KeyError(
            f"Requested column '{requested}' is ambiguous. Matching columns: {candidates}"
        )
    raise KeyError(
        f"Column '{requested}' not found. Available columns: {list(df.columns)}"
    )

def list_drive_children(service, folder_id: str):
    page_token = None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType, size, modifiedTime, parents, driveId)",
            pageSize=1000,
            pageToken=page_token,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
        ).execute()
        for item in resp.get("files", []):
            yield item
        page_token = resp.get("nextPageToken")
        if not page_token:
            break

def iter_drive_tree(service, root_folder_id: str):
    queue = deque([{
        "folder_id": root_folder_id,
        "path": "",
        "level": 0,
        "dataset_folder_id": None,
        "dataset_folder_name": None,
    }])

    while queue:
        state = queue.popleft()
        for item in list_drive_children(service, state["folder_id"]):
            is_folder = item.get("mimeType") == FOLDER_MIME
            child_path = f"{state['path']}/{item['name']}" if state["path"] else item["name"]

            if state["level"] == 0 and is_folder:
                dataset_folder_id = item["id"]
                dataset_folder_name = item["name"]
            else:
                dataset_folder_id = state["dataset_folder_id"]
                dataset_folder_name = state["dataset_folder_name"]

            row = {
                "id": item.get("id"),
                "name": item.get("name"),
                "mime_type": item.get("mimeType"),
                "is_folder": is_folder,
                "size_bytes": int(item.get("size", 0)) if str(item.get("size", "")).isdigit() else 0,
                "modified_time": item.get("modifiedTime"),
                "parents": item.get("parents", []),
                "path": child_path,
                "level": state["level"] + 1,
                "dataset_folder_id": dataset_folder_id,
                "dataset_folder_name": dataset_folder_name,
            }
            yield row

            if is_folder:
                queue.append({
                    "folder_id": item["id"],
                    "path": child_path,
                    "level": state["level"] + 1,
                    "dataset_folder_id": dataset_folder_id,
                    "dataset_folder_name": dataset_folder_name,
                })

def drive_items_to_dataframe(rows) -> pd.DataFrame:
    df = pd.DataFrame(list(rows))
    if df.empty:
        return pd.DataFrame(columns=[
            "id", "name", "mime_type", "is_folder", "size_bytes", "modified_time",
            "parents", "path", "level", "dataset_folder_id", "dataset_folder_name"
        ])
    return df

def load_worksheet_as_dataframe(sheets_service, sheet_id: str, worksheet_name: str) -> pd.DataFrame:
    meta = sheets_service.spreadsheets().get(spreadsheetId=sheet_id).execute()
    sheet_names = [s["properties"]["title"] for s in meta.get("sheets", [])]
    if worksheet_name not in sheet_names:
        raise ValueError(f"Worksheet '{worksheet_name}' not found. Available: {sheet_names}")

    resp = sheets_service.spreadsheets().values().get(
        spreadsheetId=sheet_id,
        range=f"'{worksheet_name}'"
    ).execute()

    values = resp.get("values", [])
    if not values:
        return pd.DataFrame()

    header = values[0]
    rows = values[1:]
    padded = []
    for row in rows:
        if len(row) < len(header):
            row = row + [""] * (len(header) - len(row))
        else:
            row = row[:len(header)]
        padded.append(row)

    return pd.DataFrame(padded, columns=header)

def build_match_candidates(ref_df: pd.DataFrame, drive_df: pd.DataFrame, dataset_code_col: str) -> pd.DataFrame:
    ref = ref_df.copy()
    ref[dataset_code_col] = ref[dataset_code_col].fillna("").astype(str).str.strip()
    ref["_ref_row_id"] = range(len(ref))
    ref["_dataset_code_norm"] = ref[dataset_code_col].map(lambda x: analyze_name(x)["core_norm"])

    top_level_folders = drive_df[(drive_df["level"] == 1) & (drive_df["is_folder"])].copy()
    if top_level_folders.empty:
        return pd.DataFrame(columns=[
            "_ref_row_id", dataset_code_col, "_dataset_code_norm",
            "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])

    candidate_rows = []
    for _, ref_row in ref.iterrows():
        dataset_code = ref_row[dataset_code_col]
        if not str(dataset_code).strip():
            continue

        for _, folder_row in top_level_folders.iterrows():
            score, rule = score_folder_match(dataset_code, folder_row["name"])
            if score is not None:
                candidate_rows.append({
                    "_ref_row_id": ref_row["_ref_row_id"],
                    dataset_code_col: dataset_code,
                    "_dataset_code_norm": ref_row["_dataset_code_norm"],
                    "folder_id": folder_row["id"],
                    "dataset_folder_name": folder_row["name"],
                    "match_score": score,
                    "match_rule": rule,
                })

    matches = pd.DataFrame(candidate_rows)
    if matches.empty:
        return pd.DataFrame(columns=[
            "_ref_row_id", dataset_code_col, "_dataset_code_norm",
            "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])

    return matches.sort_values(
        ["_ref_row_id", "match_score", "dataset_folder_name"],
        ascending=[True, False, True]
    ).reset_index(drop=True)

def summarize_download_status(
    drive_df: pd.DataFrame,
    ref_df: pd.DataFrame,
    dataset_code_col: str,
    quality_flag_col: str,
):
    if dataset_code_col not in ref_df.columns:
        raise KeyError(f"Column '{dataset_code_col}' not found in worksheet columns: {list(ref_df.columns)}")
    if quality_flag_col not in ref_df.columns:
        raise KeyError(f"Column '{quality_flag_col}' not found in worksheet columns: {list(ref_df.columns)}")

    ref = ref_df.copy()
    ref[dataset_code_col] = ref[dataset_code_col].fillna("").astype(str).str.strip()
    ref["_ref_row_id"] = range(len(ref))
    ref["_dataset_code_norm"] = ref[dataset_code_col].map(lambda x: analyze_name(x)["core_norm"])
    ref["is_high_quality"] = ref[quality_flag_col].map(parse_quality_flag)

    matches = build_match_candidates(ref, drive_df, dataset_code_col)

    if matches.empty:
        best_matches = pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])
        match_counts = pd.DataFrame(columns=["_ref_row_id", "match_count"])
    else:
        best_matches = (
            matches
            .sort_values(["_ref_row_id", "match_score", "dataset_folder_name"], ascending=[True, False, True])
            .drop_duplicates("_ref_row_id", keep="first")
            .copy()
        )
        match_counts = (
            matches.groupby("_ref_row_id")
            .agg(match_count=("dataset_folder_name", "nunique"))
            .reset_index()
        )

    file_stats = (
        drive_df[~drive_df["is_folder"] & drive_df["dataset_folder_name"].notna()]
        .groupby("dataset_folder_name", dropna=False)
        .agg(downloaded_files=("id", "count"), total_size_bytes=("size_bytes", "sum"))
        .reset_index()
    )

    summary = ref.merge(best_matches, on="_ref_row_id", how="left")
    summary = summary.merge(match_counts, on="_ref_row_id", how="left")
    summary = summary.merge(file_stats, on="dataset_folder_name", how="left")

    summary["downloaded"] = summary["dataset_folder_name"].notna()
    summary["match_count"] = summary["match_count"].fillna(0).astype(int)
    summary["ambiguous_match"] = summary["match_count"] > 1
    summary["downloaded_files"] = summary["downloaded_files"].fillna(0).astype(int)
    summary["total_size_bytes"] = summary["total_size_bytes"].fillna(0).astype("int64")

    duplicate_dataset_codes = (
        ref[ref["_dataset_code_norm"] != ""]
        .groupby("_dataset_code_norm", as_index=False)
        .agg(
            duplicate_rows=("_ref_row_id", "count"),
            dataset_codes=(dataset_code_col, lambda s: " | ".join(sorted({str(v) for v in s if str(v).strip()}))),
        )
    )
    duplicate_dataset_codes = duplicate_dataset_codes[duplicate_dataset_codes["duplicate_rows"] > 1].copy()

    unmatched = summary.loc[
        ~summary["downloaded"],
        [dataset_code_col, quality_flag_col, "is_high_quality", "_dataset_code_norm"]
    ].copy()

    if matches.empty:
        duplicate_matches = pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule",
            "candidate_folder_count", dataset_code_col, quality_flag_col, "is_high_quality"
        ])
    else:
        dup_ref_ids = (
            matches.groupby("_ref_row_id")["dataset_folder_name"]
            .nunique()
            .reset_index(name="candidate_folder_count")
        )
        dup_ref_ids = dup_ref_ids[dup_ref_ids["candidate_folder_count"] > 1][["_ref_row_id", "candidate_folder_count"]]

        duplicate_matches = matches.merge(dup_ref_ids, on="_ref_row_id", how="inner")
        duplicate_matches = duplicate_matches.merge(
            ref[["_ref_row_id", dataset_code_col, quality_flag_col, "is_high_quality"]],
            on="_ref_row_id",
            how="left"
        )
        duplicate_matches = duplicate_matches.sort_values(
            [dataset_code_col, "match_score", "dataset_folder_name"],
            ascending=[True, False, True]
        ).reset_index(drop=True)

    keep_cols = [col for col in summary.columns if col != "_ref_row_id"]
    return summary[keep_cols], matches, unmatched, duplicate_dataset_codes, duplicate_matches

def build_file_manifest(
    drive_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    dataset_code_col: str,
    quality_flag_col: str,
    high_quality_only: bool = True,
) -> pd.DataFrame:
    files = drive_df[~drive_df["is_folder"] & drive_df["dataset_folder_name"].notna()].copy()

    attach_cols = [
        dataset_code_col, quality_flag_col, "is_high_quality", "dataset_folder_name",
        "match_rule", "match_score", "ambiguous_match"
    ]
    attach = summary_df.loc[summary_df["downloaded"], attach_cols].drop_duplicates()
    manifest = files.merge(attach, on="dataset_folder_name", how="inner")

    ordered_cols = [
        dataset_code_col, quality_flag_col, "is_high_quality", "dataset_folder_name",
        "match_rule", "match_score", "ambiguous_match",
        "path", "size_bytes", "modified_time", "id", "mime_type"
    ]
    manifest = manifest[ordered_cols].sort_values([dataset_code_col, "path"], na_position="last")

    if high_quality_only:
        manifest = manifest[manifest["is_high_quality"].fillna(False)].copy()

    return manifest.reset_index(drop=True)

def filter_summary_by_quality(summary_df: pd.DataFrame, high_quality_only: bool) -> pd.DataFrame:
    if not high_quality_only:
        return summary_df.copy()
    return summary_df[summary_df["is_high_quality"].fillna(False)].copy()

def collapse_unique_strings(series: pd.Series) -> str:
    vals = sorted({str(v).strip() for v in series if str(v).strip()})
    return " | ".join(vals)

def build_dataset_geojson_listing(
    drive_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    dataset_code_col: str,
    quality_flag_col: str,
) -> pd.DataFrame:
    files = drive_df[~drive_df["is_folder"] & drive_df["dataset_folder_name"].notna()].copy()
    files["name_lower"] = files["name"].fillna("").astype(str).str.lower()

    aoi_files = files[files["name_lower"].str.endswith("_aoi.geojson")].copy()
    ref_files = files[
        files["name_lower"].str.endswith("_ref.geojson") |
        files["name_lower"].str.endswith("_hotosm.geojson")
    ].copy()

    aoi_agg = (
        aoi_files.groupby("dataset_folder_name", as_index=False)
        .agg(
            aoi_file_name=("name", collapse_unique_strings),
            aoi_file_path=("path", collapse_unique_strings),
            aoi_file_count=("id", "count"),
        )
    )
    ref_agg = (
        ref_files.groupby("dataset_folder_name", as_index=False)
        .agg(
            reference_file_name=("name", collapse_unique_strings),
            reference_file_path=("path", collapse_unique_strings),
            reference_file_count=("id", "count"),
        )
    )

    geo = (
        summary_df.merge(aoi_agg, on="dataset_folder_name", how="left")
                  .merge(ref_agg, on="dataset_folder_name", how="left")
                  .copy()
    )

    geo["aoi_file_count"] = geo["aoi_file_count"].fillna(0).astype(int)
    geo["reference_file_count"] = geo["reference_file_count"].fillna(0).astype(int)
    geo["has_aoi_file"] = geo["aoi_file_count"] > 0
    geo["has_reference_file"] = geo["reference_file_count"] > 0

    ordered_cols = [
        dataset_code_col,
        quality_flag_col,
        "is_high_quality",
        "dataset_folder_name",
        "aoi_file_name",
        "reference_file_name",
        "aoi_file_count",
        "reference_file_count",
        "has_aoi_file",
        "has_reference_file",
        "aoi_file_path",
        "reference_file_path",
        "match_rule",
        "match_score",
        "ambiguous_match",
    ]
    existing_cols = [c for c in ordered_cols if c in geo.columns]
    return geo[existing_cols].sort_values(
        [dataset_code_col, "dataset_folder_name"],
        na_position="last"
    ).reset_index(drop=True)

def preview_quality_values(df: pd.DataFrame, quality_col: str, n: int = 20) -> pd.DataFrame:
    vals = (
        df[quality_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .value_counts(dropna=False)
        .head(n)
        .rename_axis("quality_value")
        .reset_index(name="count")
    )
    return vals

def print_workflow_summary(
    summary: pd.DataFrame,
    unmatched: pd.DataFrame,
    duplicate_dataset_codes: pd.DataFrame,
    duplicate_matches: pd.DataFrame,
    drive_df: pd.DataFrame,
) -> None:
    total = len(summary)
    downloaded = int(summary["downloaded"].fillna(False).sum()) if "downloaded" in summary.columns else 0
    ambiguous = int(summary["ambiguous_match"].fillna(False).sum()) if "ambiguous_match" in summary.columns else 0
    print("Workflow summary:")
    print(f"  Worksheet rows           : {total}")
    print(f"  Matched/downloaded rows  : {downloaded}")
    print(f"  Unmatched rows           : {len(unmatched)}")
    print(f"  Duplicate dataset codes  : {len(duplicate_dataset_codes)}")
    print(f"  Ambiguous folder matches : {len(duplicate_matches)}")
    print(f"  Drive inventory rows     : {len(drive_df)}")


In [6]:
# --- Patch the matching helpers so resolved column names work correctly ---
# Paste this into a new cell and run it BEFORE the main workflow cell.

def build_match_candidates(ref_df: pd.DataFrame, drive_df: pd.DataFrame, dataset_code_col: str) -> pd.DataFrame:
    ref = ref_df.copy()
    ref[dataset_code_col] = ref[dataset_code_col].fillna("").astype(str).str.strip()
    ref["_ref_row_id"] = range(len(ref))

    top_level_folders = drive_df[(drive_df["level"] == 1) & (drive_df["is_folder"])].copy()
    if top_level_folders.empty:
        return pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])

    candidate_rows = []
    for _, ref_row in ref.iterrows():
        dataset_code = ref_row[dataset_code_col]
        if not str(dataset_code).strip():
            continue

        for _, folder_row in top_level_folders.iterrows():
            score, rule = score_folder_match(dataset_code, folder_row["name"])
            if score is not None:
                candidate_rows.append({
                    "_ref_row_id": ref_row["_ref_row_id"],
                    "folder_id": folder_row["id"],
                    "dataset_folder_name": folder_row["name"],
                    "match_score": score,
                    "match_rule": rule,
                })

    matches = pd.DataFrame(candidate_rows)
    if matches.empty:
        return pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])

    matches = matches.sort_values(
        ["_ref_row_id", "match_score", "dataset_folder_name"],
        ascending=[True, False, True]
    ).reset_index(drop=True)
    return matches


def summarize_download_status(
    drive_df: pd.DataFrame,
    ref_df: pd.DataFrame,
    dataset_code_col: str,
    quality_flag_col: str,
):
    if dataset_code_col not in ref_df.columns:
        raise KeyError(f"Column '{dataset_code_col}' not found in worksheet columns: {list(ref_df.columns)}")
    if quality_flag_col not in ref_df.columns:
        raise KeyError(f"Column '{quality_flag_col}' not found in worksheet columns: {list(ref_df.columns)}")

    ref = ref_df.copy()
    ref[dataset_code_col] = ref[dataset_code_col].fillna("").astype(str).str.strip()
    ref["_ref_row_id"] = range(len(ref))
    ref["_dataset_code_norm"] = ref[dataset_code_col].map(lambda x: analyze_name(x)["core_norm"])
    ref["is_high_quality"] = ref[quality_flag_col].map(parse_quality_flag)

    matches = build_match_candidates(ref, drive_df, dataset_code_col)

    if matches.empty:
        best_matches = pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule"
        ])
        match_counts = pd.DataFrame(columns=["_ref_row_id", "match_count"])
    else:
        best_matches = (
            matches
            .sort_values(["_ref_row_id", "match_score", "dataset_folder_name"], ascending=[True, False, True])
            .drop_duplicates("_ref_row_id", keep="first")
            .copy()
        )

        match_counts = (
            matches.groupby("_ref_row_id")
            .agg(match_count=("dataset_folder_name", "nunique"))
            .reset_index()
        )

    file_stats = (
        drive_df[~drive_df["is_folder"] & drive_df["dataset_folder_name"].notna()]
        .groupby("dataset_folder_name", dropna=False)
        .agg(downloaded_files=("id", "count"), total_size_bytes=("size_bytes", "sum"))
        .reset_index()
    )

    summary = ref.merge(best_matches, on="_ref_row_id", how="left")
    summary = summary.merge(match_counts, on="_ref_row_id", how="left")
    summary = summary.merge(file_stats, on="dataset_folder_name", how="left")

    summary["downloaded"] = summary["dataset_folder_name"].notna()
    summary["match_count"] = summary["match_count"].fillna(0).astype(int)
    summary["ambiguous_match"] = summary["match_count"] > 1
    summary["downloaded_files"] = summary["downloaded_files"].fillna(0).astype(int)
    summary["total_size_bytes"] = summary["total_size_bytes"].fillna(0).astype("int64")

    duplicate_dataset_codes = (
        ref[ref["_dataset_code_norm"] != ""]
        .groupby("_dataset_code_norm", as_index=False)
        .agg(
            duplicate_rows=("_ref_row_id", "count"),
            dataset_codes=(dataset_code_col, lambda s: " | ".join(sorted({str(v) for v in s if str(v).strip()}))),
        )
    )
    duplicate_dataset_codes = duplicate_dataset_codes[duplicate_dataset_codes["duplicate_rows"] > 1].copy()

    unmatched = summary.loc[
        ~summary["downloaded"],
        [dataset_code_col, quality_flag_col, "is_high_quality", "_dataset_code_norm"]
    ].copy()

    if matches.empty:
        duplicate_matches = pd.DataFrame(columns=[
            "_ref_row_id", "folder_id", "dataset_folder_name", "match_score", "match_rule",
            "candidate_folder_count", dataset_code_col, quality_flag_col, "is_high_quality"
        ])
    else:
        dup_ref_ids = (
            matches.groupby("_ref_row_id")["dataset_folder_name"]
            .nunique()
            .reset_index(name="candidate_folder_count")
        )
        dup_ref_ids = dup_ref_ids[dup_ref_ids["candidate_folder_count"] > 1][["_ref_row_id", "candidate_folder_count"]]

        duplicate_matches = matches.merge(dup_ref_ids, on="_ref_row_id", how="inner")
        duplicate_matches = duplicate_matches.merge(
            ref[["_ref_row_id", dataset_code_col, quality_flag_col, "is_high_quality"]],
            on="_ref_row_id",
            how="left"
        )
        duplicate_matches = duplicate_matches.sort_values(
            [dataset_code_col, "match_score", "dataset_folder_name"],
            ascending=[True, False, True]
        ).reset_index(drop=True)

    keep_cols = [col for col in summary.columns if col != "_ref_row_id"]
    return summary[keep_cols], matches, unmatched, duplicate_dataset_codes, duplicate_matches

In [7]:
# Main workflow
drive_df = drive_items_to_dataframe(iter_drive_tree(drive_service, CONFIG["drive_folder_id"]))
ref_df = load_worksheet_as_dataframe(sheets_service, CONFIG["sheet_id"], CONFIG["worksheet"])

dataset_code_col = resolve_column_name(ref_df, CONFIG["dataset_code_col"])
quality_col = resolve_column_name(ref_df, CONFIG["quality_col"])

print(f"Using dataset code column: {dataset_code_col}")
print(f"Using quality flag column: {quality_col}")

summary, matches, unmatched, duplicate_dataset_codes, duplicate_matches = summarize_download_status(
    drive_df=drive_df,
    ref_df=ref_df,
    dataset_code_col=dataset_code_col,
    quality_flag_col=quality_col,
)

file_manifest = build_file_manifest(
    drive_df=drive_df,
    summary_df=summary,
    dataset_code_col=dataset_code_col,
    quality_flag_col=quality_col,
    high_quality_only=CONFIG["high_quality_only"],
)

summary.to_csv(SUMMARY_OUT, index=False)
file_manifest.to_csv(FILES_OUT, index=False)
unmatched.to_csv(UNMATCHED_OUT, index=False)
duplicate_matches.to_csv(DUPLICATE_MATCHES_OUT, index=False)
duplicate_dataset_codes.to_csv(DUPLICATE_CODES_OUT, index=False)

print(f"Wrote summary: {SUMMARY_OUT} ({len(summary)} rows)")
print(f"Wrote file manifest: {FILES_OUT} ({len(file_manifest)} rows)")
print(f"Wrote unmatched datasets: {UNMATCHED_OUT} ({len(unmatched)} rows)")
print(f"Wrote duplicate matches: {DUPLICATE_MATCHES_OUT} ({len(duplicate_matches)} rows)")
print(f"Wrote duplicate dataset codes: {DUPLICATE_CODES_OUT} ({len(duplicate_dataset_codes)} rows)")
print_workflow_summary(summary, unmatched, duplicate_dataset_codes, duplicate_matches, drive_df)


Using dataset code column: Dataset code
Using quality flag column: Suitable (yes/N)
Wrote summary: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/reference_download_status.csv (478 rows)
Wrote file manifest: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/reference_download_file_manifest.csv (343 rows)
Wrote unmatched datasets: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/reference_unmatched_datasets.csv (145 rows)
Wrote duplicate matches: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/reference_duplicate_matches.csv (36 rows)
Wrote duplicate dataset codes: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/reference_duplicate_dataset_codes.csv (38 rows)
Workflow summary:
  Worksheet 

## Optional diagnostics

This cell helps verify that the worksheet columns were resolved correctly and shows a quick preview of dataset-code and quality-flag values.


In [8]:
# Optional diagnostics
print(f"Worksheet: {CONFIG['worksheet']}")
print(f"Rows: {len(ref_df)}")
print("\nAvailable columns:")
for i, c in enumerate(ref_df.columns, start=1):
    print(f"{i:>2}. {c}")

print("\nResolved columns:")
print(f"  Dataset code column -> {dataset_code_col}")
print(f"  Quality flag column -> {quality_col}")

print("\nSample dataset codes:")
display(
    ref_df[[dataset_code_col]]
    .rename(columns={dataset_code_col: "dataset_code_sample"})
    .head(10)
)

print("\nTop quality flag values:")
display(preview_quality_values(ref_df, quality_col, n=20))

missing_dataset_codes = (
    ref_df[dataset_code_col].fillna("").astype(str).str.strip().eq("").sum()
)
missing_quality_flags = (
    ref_df[quality_col].fillna("").astype(str).str.strip().eq("").sum()
)

print("\nMissing value check:")
print(f"  Blank dataset codes : {missing_dataset_codes}")
print(f"  Blank quality flags : {missing_quality_flags}")


Worksheet: Reference Dataset overview
Rows: 478

Available columns:
 1. Country
 2. City
 3. Description
 4. Suitable (yes/N)
 5. Quality tag
 6. Dataset code
 7. Dataset copied to gdrive
 8. Coverage
 9. Year (data published)
10. Year (of the image)
11. Format
12. Contact / source
13. Link
14. Comments
15. Data links

Resolved columns:
  Dataset code column -> Dataset code
  Quality flag column -> Suitable (yes/N)

Sample dataset codes:


,dataset_code_sample
0,
1,ssd-juba
2,rwa-kigali
3,zaf-johanesburg-capetown
4,ner-mli-kandadji
5,-
6,dom-dominica
7,lca-saint lucia
8,cvg-saint vincent grenadines
9,grd-grenada



Top quality flag values:


,quality_value,count
0,N,312
1,yes,160
2,not yet,2



Missing value check:
  Blank dataset codes : 4
  Blank quality flags : 4


## Optional AOI/reference GeoJSON listing

This cell creates a dataset-level table showing AOI files (`*_aoi.geojson`) and reference files (`*_ref.geojson` or `*_hotosm.geojson`) for the matched datasets.


In [9]:
# Optional AOI/reference GeoJSON listing
matched_summary = summary.loc[summary["downloaded"]].copy()
matched_summary = filter_summary_by_quality(
    matched_summary,
    CONFIG["high_quality_only"]
)

dataset_geojson_list = build_dataset_geojson_listing(
    drive_df=drive_df,
    summary_df=matched_summary,
    dataset_code_col=dataset_code_col,
    quality_flag_col=quality_col,
)

dataset_geojson_list.to_csv(GEOJSON_LIST_OUT, index=False)
print(f"Wrote AOI/reference dataset list: {GEOJSON_LIST_OUT} ({len(dataset_geojson_list)} rows)")
display(dataset_geojson_list.head(30))


Wrote AOI/reference dataset list: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/reference_inventory/dataset_aoi_reference_list.csv (144 rows)


,Dataset code,Suitable (yes/N),is_high_quality,dataset_folder_name,aoi_file_name,reference_file_name,aoi_file_count,reference_file_count,has_aoi_file,has_reference_file,aoi_file_path,reference_file_path,match_rule,match_score,ambiguous_match
0,ant-curacao,yes,True,ant-curacao,curacao_aoi.geojson,curacao_hotosm.geojson,1,1,True,True,ant-curacao/aoi/curacao_aoi.geojson,ant-curacao/vector/curacao_hotosm.geojson,exact_normalized,100.0,False
1,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
2,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
3,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
4,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
5,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
6,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
7,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
8,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
9,bgd-rohingya,yes,True,bgd-rohingya,rohingya_3939_aoi.geojson | rohingya_4003_aoi....,rohingya_3939_hotosm.geojson | rohingya_4003_h...,33,33,True,True,bgd-rohingya/aoi/rohingya_3939_aoi.geojson | b...,bgd-rohingya/vector/rohingya_3939_hotosm.geojs...,exact_normalized,100.0,False
